# Path2Space Benchmark (multi-slide, Protocol A)

Strict Path2Space reproduction using this repository's HEST data and gene list.
Multi-slide supervision: train on two sections, validate on a third, test on the fourth.

Source-faithful core (unchanged from upstream):
- `func/ctrans_model/ctranspath.py::CTransPath` -- frozen 768-d feature extractor
- `model_MLP.py::MLP_regression_relu_two` -- Linear(768,768)->ReLU->Dropout(0.2)->Linear(768,G)->ReLU
- MSE loss, Adam lr=1e-4, spot-batch 32, upstream `training_epoch()` / `predict()`
- best-validation checkpoint (see README_MORPHOFM_ADAPTATION.md; upstream keeps only
  the final-epoch weights, which the rest of this benchmark does not do)

Adapted data surface only: HEST whole-slide reading, spot barcode alignment,
`common_genes.txt` gene panel, fixed slide-level splits, prediction export.


In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Optional OpenSlide
try:
    import openslide
    HAS_OPENSLIDE = True
except ImportError:
    HAS_OPENSLIDE = False
    print("OpenSlide not found, using PIL fallback (may be slow).")

ROOT = Path(os.environ.get("MORPHO_VC_ROOT", "../")).expanduser().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "benchmark") not in sys.path:
    sys.path.insert(0, str(ROOT / "benchmark"))

P2S_ROOT = ROOT / "benchmark" / "Path2Space"
P2S_REGRESSION_SRC = P2S_ROOT / "scripts" / "1.ST_prediction" / "1.2.Regression"
sys.path.append(str(P2S_REGRESSION_SRC))

from model_MLP import MLP_regression_relu_two, training_epoch, predict

from hest_dataset import (
    extract_path2space_features,
    path2space_target_matrix,
    HESTPath2SpaceDataset,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# IDs (multi-slide Protocol A defaults; patched per fold by the runner)
train_ids = ["INT25", "INT26"]
val_ids = ["INT27"]
test_ids = ["INT28"]
all_ids = train_ids + val_ids + test_ids

# Paths
result_dir = ROOT / "benchmark" / "results" / "path2space"
result_dir.mkdir(parents=True, exist_ok=True)

feature_cache_dir = ROOT / "results" / "prostate_hest" / "path2space" / "features"
feature_cache_dir.mkdir(parents=True, exist_ok=True)

spatial_dir = ROOT / "data" / "spatial_data"
wsi_dir = ROOT / "data" / "hest_data" / "wsis"
common_gene_path = spatial_dir / "common_genes.txt"

ctranspath_weights = ROOT / "benchmark" / "HiST" / "resource" / "ctranspath.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

P2S_TILE_SIZE = 224
P2S_WINDOW = "tile_px"
P2S_RESIZE_TO = 224
USE_MACENKO = False

print("Result dir:", result_dir)
print(f"Train: {train_ids} | Val: {val_ids} | Test: {test_ids}")
print("Device:", device)


In [ ]:
# ===== Helpers =====


def read_gene_list(path: Path):
    genes = path.read_text(encoding="utf-8").splitlines()
    return [g.strip() for g in genes if g.strip()]


print("Helpers ready.")


In [ ]:
# ===== Step 1: Path2Space CTransPath features (one 768-d vector per in-tissue spot) =====

genes = read_gene_list(common_gene_path)
print(f"Genes: {len(genes)}")

feature_bundle = extract_path2space_features(
    all_ids,
    hest_dir=ROOT / "data" / "hest_data",
    spatial_dir=spatial_dir,
    upstream_root=P2S_ROOT,
    ctranspath_weights=ctranspath_weights,
    tile_size=P2S_TILE_SIZE,
    window=P2S_WINDOW,
    resize_to=P2S_RESIZE_TO,
    use_macenko=USE_MACENKO,
    feature_cache_dir=feature_cache_dir,
    batch_size=128,
    device=str(device),
    verbose=True,
)

for sid in all_ids:
    feats = feature_bundle[sid]["features"]
    print(f"{sid}: features {feats.shape} | finite={np.isfinite(feats).all()}")


In [ ]:
# ===== Step 2: Targets + train/valid/test datasets =====

import anndata as ad

targets = {}
for sid in all_ids:
    adata = ad.read_h5ad(spatial_dir / f"{sid}.h5ad")
    targets[sid] = path2space_target_matrix(adata, genes)
    print(f"{sid}: target {targets[sid].shape}")

train_set = HESTPath2SpaceDataset({s: feature_bundle[s] for s in train_ids},
                                  {s: targets[s] for s in train_ids}, genes)
valid_set = HESTPath2SpaceDataset({s: feature_bundle[s] for s in val_ids},
                                  {s: targets[s] for s in val_ids}, genes)
test_set = HESTPath2SpaceDataset({s: feature_bundle[s] for s in test_ids},
                                 {s: targets[s] for s in test_ids}, genes)
print(f"train_set: {len(train_set)} | valid_set: {len(valid_set)} | test_set: {len(test_set)}")


In [ ]:
# ===== Step 3: Train upstream MLP_regression_relu_two with a real validation slide =====

N_INPUTS = 768
N_HIDDENS = 768
DROPOUT = 0.2
SPOT_BATCH_SIZE = 32
LEARNING_RATE = 1.0e-4
MAX_EPOCHS = 200
PATIENCE = MAX_EPOCHS // 10

bias_init = torch.nn.Parameter(
    torch.Tensor(np.mean([train_set[i][1].detach().cpu().numpy() for i in range(len(train_set))],
                         axis=0)).to(device)
)

model = MLP_regression_relu_two(N_INPUTS, N_HIDDENS, len(genes), DROPOUT, bias_init)
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(" --- fit --- ")

# Upstream `fit()` keeps only the final-epoch weights. HisToGene, iStar and sCellST all
# select the best-validation checkpoint in this benchmark, so this loop calls upstream's
# own `training_epoch()` / `predict()` unchanged and adds only the checkpoint snapshot.
import copy as _copy

train_loss, train_coef, valid_loss, valid_coef = [], [], [], []
best_state = None
best_val = -float("inf")
best_epoch = -1
epoch_since_best = 0
valid_labels = valid_preds = None

for epoch in range(MAX_EPOCHS):
    epoch_since_best += 1
    tl, tc = training_epoch(model, optimizer, train_set, SPOT_BATCH_SIZE)
    vl, vc, valid_labels, valid_preds = predict(model, valid_set)

    print(f"{epoch}, train_loss: {tl:.4f}, coef: {tc:.4f},\tvalid_loss: {vl:.4f}, coef: {vc:.4f}")
    train_loss.append(tl); train_coef.append(tc); valid_loss.append(vl); valid_coef.append(vc)

    if vc > best_val:
        best_val = vc
        best_epoch = epoch
        best_state = _copy.deepcopy(model.state_dict())
        epoch_since_best = 0

    if epoch_since_best == PATIENCE:
        print(f"Early stopping at epoch {epoch + 1}")
        break

model.load_state_dict(best_state)
print(f"\nBest epoch {best_epoch} | valid mean gene Pearson {best_val:.4f} (restored)")

torch.save(model.state_dict(), result_dir / "model_trained.pth")
np.savetxt(result_dir / "train_valid_loss.txt",
           np.array((train_loss, valid_loss, train_coef, valid_coef)).T, fmt="%.6f")
with open(result_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump({
        "method": "Path2Space",
        "protocol": "rotating_heldout_multi_slide",
        "train_slide_ids": train_ids, "val_slide_ids": val_ids, "test_slide_ids": test_ids,
        "n_inputs": N_INPUTS, "n_hiddens": N_HIDDENS, "dropout": DROPOUT,
        "spot_batch_size": SPOT_BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "seed": SEED,
        "tile_size": P2S_TILE_SIZE, "window": P2S_WINDOW, "resize_to": P2S_RESIZE_TO,
        "use_macenko": USE_MACENKO,
        "best_epoch": best_epoch, "best_valid_mean_gene_pearson": float(best_val),
        "n_genes": len(genes),
    }, f, ensure_ascii=False, indent=2)
print("Saved:", result_dir / "model_trained.pth")


In [ ]:
# ===== Step 4: Evaluation + Save pred_bag/true_bag + metrics =====

model.eval()

# batch_size=1: upstream forward ends with torch.mean(x, dim=0), which collapses the
# leading axis rather than averaging over tiles -- correct only for one sample at a time,
# exactly how upstream's own loop calls it.
pred_bag_list = []
true_bag_list = []
with torch.no_grad():
    for sid in test_ids:
        base = HESTPath2SpaceDataset({sid: feature_bundle[sid]}, {sid: targets[sid]}, genes)
        loader = DataLoader(base, batch_size=1, shuffle=False, num_workers=0)
        preds = []
        for x, y in loader:
            preds.append(model(x.float().to(device))[0].detach().cpu().numpy())
        pred_vals = np.stack(preds, axis=0)
        true_vals = targets[sid]
        pred_bag_list.append(pred_vals)
        true_bag_list.append(true_vals)
        print(f"{sid}: pred {pred_vals.shape} | true {true_vals.shape}")

pred_bag = np.concatenate(pred_bag_list, axis=0).astype(np.float32)
true_bag = np.concatenate(true_bag_list, axis=0).astype(np.float32)
print("Prediction Shape:", pred_bag.shape)


def pearson_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if a.size == 0 or b.size == 0:
        return np.nan
    if np.all(a == a[0]) or np.all(b == b[0]):
        return np.nan
    a = a - a.mean()
    b = b - b.mean()
    denom = np.sqrt((a * a).sum()) * np.sqrt((b * b).sum())
    if denom == 0:
        return np.nan
    return float((a * b).sum() / denom)


mae = float(np.mean(np.abs(pred_bag - true_bag)))
rmse = float(np.sqrt(np.mean((pred_bag - true_bag) ** 2)))

corrs = [pearson_corr(pred_bag[:, i], true_bag[:, i]) for i in range(pred_bag.shape[1])]
valid = [(i, c) for i, c in enumerate(corrs) if not np.isnan(c)]
mean_pearson = float(np.mean([c for _, c in valid])) if valid else float("nan")
best_gene_idx, best_gene_corr = max(valid, key=lambda x: x[1]) if valid else (None, None)
best_gene_name = genes[best_gene_idx] if best_gene_idx is not None else "NA"

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Mean Pearson (gene-wise): {mean_pearson:.4f}")
print(f"Best Gene: {best_gene_name} (Pearson={best_gene_corr:.4f})")

np.save(result_dir / "pred_bag.npy", pred_bag)
np.save(result_dir / "true_bag.npy", true_bag)
with open(result_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "MAE": mae, "RMSE": rmse, "Mean_Pearson": mean_pearson,
        "Best_Gene": best_gene_name,
        "Best_Pearson": float(best_gene_corr) if best_gene_corr is not None else None,
    }, f, ensure_ascii=False, indent=2)
print("Saved pred_bag/true_bag/metrics.json")


In [ ]:
# ===== Step 5: Visualization (01-style, top-4 genes on the test slide) =====

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

VIS_ID = test_ids[0]
adata_vis = ad.read_h5ad(spatial_dir / f"{VIS_ID}.h5ad")
mask_vis = adata_vis.obs["in_tissue"].to_numpy().astype(bool)
xy = adata_vis.obs[["pxl_col_in_fullres", "pxl_row_in_fullres"]].to_numpy()[mask_vis]

pred_vals = pred_bag[: mask_vis.sum()]
true_vals = true_bag[: mask_vis.sum()]
corrs_vis = np.array([pearson_corr(pred_vals[:, i], true_vals[:, i]) for i in range(pred_vals.shape[1])])
valid_idx = np.where(np.isfinite(corrs_vis))[0]
top4 = valid_idx[np.argsort(corrs_vis[valid_idx])[::-1][:4]]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for j, gi in enumerate(top4):
    for row, (vals, tag) in enumerate([(true_vals, "measured"), (pred_vals, "predicted")]):
        ax = axes[row, j]
        sc = ax.scatter(xy[:, 0], xy[:, 1], c=vals[:, gi], s=3, cmap="viridis")
        ax.set_title(f"{genes[gi]} ({tag})\nr={corrs_vis[gi]:.3f}")
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(sc, ax=ax, fraction=0.046)
plt.tight_layout()
out_fig = result_dir / "top4_genes_vis.png"
plt.savefig(out_fig, dpi=150)
plt.close()
print("Saved:", out_fig)
